# 01 — Data Loading & Merge

**Goal of this notebook**: turn the raw Figshare `.mat` files and Br35H "no tumor"
images into a single `metadata.csv` — the source of truth every later notebook reads
from. We deliberately do *not* load full image arrays into memory here; only paths
and labels, so this step stays fast and safe to re-run.

**Why this step matters for data integrity**: the Figshare files store the patient ID
(`cjdata.PID`) as a MATLAB string, which in the HDF5-based `.mat` format is an array of
character codes, not a plain scalar — `src/data_utils.py` handles this decoding
correctly (`_decode_matlab_string`), verified against a synthetic file matching the real
structure in `tests/test_data_utils.py`.

**Prerequisite**: raw data must already be downloaded per `data/README.md` before
running this notebook — it is not fetched automatically.

In [19]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

from src.data_utils import build_metadata

FIGSHARE_DIR = Path("../data/raw/figshare_mat")
BR35H_DIR = Path("../data/raw/br35h_no_tumor")
OUTPUT_CSV = Path("../data/processed/metadata.csv")

## If something looks off — diagnose first

**Intent**: `build_metadata()` below validates every `.mat` file before processing
(missing/renamed folders, corrupted downloads, or a wrong MATLAB format are the most
common causes of failure here) and will raise a clear, per-file error if something's
wrong — rather than crashing partway through on an unhelpful KeyError.

If it does raise, run the standalone diagnostic script for a full report across every
file (not just the first 10 in the error message):

```bash
python ../scripts/inspect_mat_files.py
```

It reports exactly which files are invalid and why (missing `cjdata` group, missing
fields, or not a valid MATLAB v7.3/HDF5 file), so you know whether to re-download,
re-extract, or just remove a handful of bad files.

## Build the unified metadata table

This scans both raw sources and writes `metadata.csv` with columns:
`image_path, label, patient_id, source_dataset`.

In [20]:
metadata = build_metadata(FIGSHARE_DIR, BR35H_DIR, OUTPUT_CSV)
print(f"Total images: {len(metadata)}")
metadata.head()

Total images: 4564


,image_path,label,patient_id,source_dataset
0,..\data\raw\figshare_mat\1.mat,meningioma,figshare_100360,figshare
1,..\data\raw\figshare_mat\10.mat,meningioma,figshare_101016,figshare
2,..\data\raw\figshare_mat\100.mat,meningioma,figshare_107494,figshare
3,..\data\raw\figshare_mat\1000.mat,pituitary,figshare_112649,figshare
4,..\data\raw\figshare_mat\1001.mat,pituitary,figshare_112649,figshare


## Sanity checks before moving on

Quick checks that the merge did what we expect — class presence, patient ID
uniqueness pattern (Figshare patients should repeat across slices, Br35H should not).

In [21]:
print("Class counts:")
print(metadata["label"].value_counts())

print("\nSource dataset counts:")
print(metadata["source_dataset"].value_counts())

print("\nSlices per Figshare patient (should mostly be >1):")
figshare_only = metadata[metadata["source_dataset"] == "figshare"]
print(figshare_only.groupby("patient_id").size().describe())

print("\nBr35H images per synthetic patient_id (should all be exactly 1):")
br35h_only = metadata[metadata["source_dataset"] == "br35h"]
assert (
    br35h_only.groupby("patient_id").size() == 1
).all(), "Br35H patient IDs should be unique per image"
print("OK — every Br35H image has its own unique group.")

Class counts:
label
no_tumor      1500
glioma        1426
pituitary      930
meningioma     708
Name: count, dtype: int64

Source dataset counts:
source_dataset
figshare    3064
br35h       1500
Name: count, dtype: int64

Slices per Figshare patient (should mostly be >1):
count    233.000000
mean      13.150215
std        7.655779
min        1.000000
25%        7.000000
50%       13.000000
75%       18.000000
max       38.000000
dtype: float64

Br35H images per synthetic patient_id (should all be exactly 1):
OK — every Br35H image has its own unique group.


Next notebook: **02_eda.ipynb** — explore class balance, patient distribution, and
sample images before we touch any modeling.